<a href="https://colab.research.google.com/github/Abrar-404/AI-ML_Practices_and_Assignments/blob/main/Module_20_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# Problem 1: Breast Cancer Classification
### Dataset: https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data

Objective

- Predict whether a tumor is:
  - Malignant (M)
  - Benign (B)

Tasks
  - Load the dataset.
  - Drop unnecessary columns (e.g., id, Unnamed: 32).
  - Convert target labels (M → 1, B → 0).
  - Split the dataset (80/20).
  - Standardize the features.
  - Train an SVM with a Linear Kernel.

Evaluate using:
 - Accuracy
 - Precision
- Recall


In [3]:
# Load the dataset.
df_cancer = pd.read_csv('data.csv')

# Drop unnecessary columns (e.g., id, Unnamed: 32).
X = df_cancer.drop(['id', 'Unnamed: 32', 'diagnosis'], axis = 1)

# Convert target labels (M → 1, B → 0)
df_cancer['diagnosis'] = df_cancer['diagnosis'].map({'M': 1, 'B': 0})

y = df_cancer['diagnosis']

# Split the dataset (80/20).
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

# Standardize the features.
num_cols = X.select_dtypes(include = ['int64', 'float64']).columns
cat_cols = X.select_dtypes(include = 'object').columns

num_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'median')),
        ('scaler', StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'most_frequent')),
        ('encoder', OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ]
)

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Train an SVM with a Linear Kernel.
svc_model = Pipeline(
    steps = [
        ('processor', preprocessor),
        ('model', SVC())
    ]
)

grid_param = {
    'model__kernel': ['linear'],
    'model__C': [0.001, 0.01, 1, 10, 50, 100]
}

best_SVC_model = GridSearchCV(
    estimator = svc_model,
    param_grid = grid_param,
    cv = 5
    )

best_SVC_model.fit(X_train,y_train)
y_pred = best_SVC_model.predict(X_test)

# Evaluate using: Accuracy, Precision, Recall
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)

print(f'Accuracy: {round(acc, 2)}, Precision: {round(prec, 2)}, Recall: {round(rec, 2)}')

Accuracy: 0.96, Precision: 1.0, Recall: 0.88


# Problem 2: Mushroom Classification
### Dataset: https://www.kaggle.com/datasets/uciml/mushroom-classification

Objective
- Predict whether a mushroom is:
  - Edible
  - Poisonous

Tasks
- Handle categorical variables.
- Encode all features.
- Train Linear SVM.
- Evaluate performance.
- Compare with Decision Tree.


In [4]:
# load dataset
df_mush = pd.read_csv('mushrooms.csv')

# Handle categorical variables.
X = df_mush.drop('class', axis = 1)
y = df_mush['class'].map({'e': 0, 'p': 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

# Handle categorical variables.
cat_cols = X.select_dtypes(include = 'object').columns

# Encode all features.
cat_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'most_frequent')),
        ('encoder', OneHotEncoder(sparse_output = False, drop = 'first', handle_unknown = 'ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('cat', cat_pipeline, cat_cols)
    ]
)

# Train Linear SVM.
svm_model = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', SVC())
    ]
)

grid_param_svm = {
    'model__kernel': ['linear'],
    'model__C': [0.001, 0.01, 1, 10]
}

grid_search_svm = GridSearchCV(
    estimator = svm_model,
    param_grid = grid_param_svm,
    cv = 5
)

grid_search_svm.fit(X_train, y_train)
y_pred_svm = grid_search_svm.predict(X_test)

# Evaluate performance.
acc_svm = accuracy_score(y_test, y_pred_svm)
prec_svm = precision_score(y_test, y_pred_svm)
rec_svm = recall_score(y_test, y_pred_svm)

print(f'Accuracy-SVM: {round(acc_svm, 2)}, Precision-SVM: {round(prec_svm, 2)}, Recall-SVM: {round(rec_svm, 2)}')

# Compare with Decision Tree.
decision_model = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model_dec', DecisionTreeClassifier(random_state=42, max_depth = None))
    ]
)

decision_model.fit(X_train, y_train)
y_pred_dec = decision_model.predict(X_test)

acc_dec = accuracy_score(y_test, y_pred_dec)
prec_dec = precision_score(y_test, y_pred_dec)
rec_dec = recall_score(y_test, y_pred_dec)

print()

print(f'Accuracy-DEC: {round(acc_dec, 2)}, Precision-DEC: {round(prec_dec, 2)}, Recall-DEC: {round(rec_dec, 2)}')

# Sanity check: how separable is this dataset really?
print(df_mush['class'].value_counts())          # verify classes aren't wildly imbalanced
print(pd.crosstab(df_mush['odor'], df_mush['class']))  # odor vs class — see near-perfect split

Accuracy-SVM: 1.0, Precision-SVM: 1.0, Recall-SVM: 1.0

Accuracy-DEC: 1.0, Precision-DEC: 1.0, Recall-DEC: 1.0
class
e    4208
p    3916
Name: count, dtype: int64
class     e     p
odor             
a       400     0
c         0   192
f         0  2160
l       400     0
m         0    36
n      3408   120
p         0   256
s         0   576
y         0   576


# Problem 3: Bank Marketing
### Dataset: https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset

Objective
- Predict whether a customer will subscribe to a term deposit.

Tasks
- Handle categorical variables.
- Feature scaling.
- Train Linear SVM.
- Print evaluation metrics.


In [5]:
# load dataset
df_bank = pd.read_csv('bank.csv')

X = df_bank.drop(['deposit', 'duration'], axis = 1)
y = df_bank['deposit'].map({'yes': 1, 'no': 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

# Handle categorical variables.
num_cols = X.select_dtypes(include = ['int64', 'float64']).columns
cat_cols = X.select_dtypes(include = 'object').columns

# Feature scaling.
num_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'median')),
        ('scaler', StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'most_frequent')),
        ('encoder', OneHotEncoder(sparse_output = False, drop = 'first', handle_unknown = 'ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ]
)

# Train Linear SVM.
svm_mod = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', SVC())
    ]
)

grid_par = [
  {
    'model__kernel' : ['linear'],
    'model__C': [0.001, 0.01, 1]
  }
]

grid_search_svm = GridSearchCV(
    estimator = svm_mod,
    param_grid = grid_par,
    cv = 3
)

grid_search_svm.fit(X_train, y_train)
y_pred = grid_search_svm.predict(X_test)

# Print evaluation metrics.
acc_sv = accuracy_score(y_test, y_pred)
prec_sv = precision_score(y_test, y_pred)
rec_sv = recall_score(y_test, y_pred)

print(f'Accuracy: {round(acc_sv, 2)}, Precision: {round(prec_sv, 2)}, Recall: {round(rec_sv, 2)}')

Accuracy: 0.69, Precision: 0.74, Recall: 0.54


# Problem 4: Heart Disease Prediction
### Dataset: https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction

Objective
- Predict whether a patient has heart disease.

Tasks
- Train four SVM models:
  - Linear
  - Polynomial
  - RBF
  - Sigmoid

Compare
- Accuracy
- Precision
- Recall


In [6]:
# load dataset
df_heart = pd.read_csv('heart.csv')

X = df_heart.drop('target', axis = 1)
y = df_heart['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

num_cols = X.select_dtypes(include = ['int64', 'float64']).columns

num_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'median')),
        ('scaler', StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_pipeline, num_cols)
    ]
)

# Train four SVM models: Linear, Polynomial, RBF, Sigmoid
svm_models = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('models', SVC())
    ]
)

kernel_grids = {
    'linear':  {'model__kernel': ['linear'], 'model__C': [0.001, 0.01, 1]},
    'poly':    {'model__kernel': ['poly'], 'model__C': [0.001, 0.01, 1], 'model__degree': [2, 3]},
    'rbf':     {'model__kernel': ['rbf'], 'model__C': [0.1, 1, 5, 10], 'model__gamma': [0.01, 1, 'scale', 'auto']},
    'sigmoid': {'model__kernel': ['sigmoid'], 'model__gamma': [0.1, 1, 'scale'], 'model__coef0': [0.0]}
}


results = {}

for name, grid in kernel_grids.items():
  model_svm = Pipeline(
      steps = [
          ('preprocessor', preprocessor),
          ('model', SVC())
      ]
  )

  grid_search = GridSearchCV(
      estimator = model_svm,
      param_grid = grid,
      cv = 3
  )

  grid_search.fit(X_train, y_train)
  y_pred = grid_search.predict(X_test)

  results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'Best Params': grid_search.best_params_
    }

  print(f"{name.upper():8s} -> Accuracy: {round(results[name]['Accuracy'],2)}, "
      f"Precision: {round(results[name]['Precision'],2)}, "
      f"Recall: {round(results[name]['Recall'],2)}")

# Optional: put results in a comparison table
results_df = pd.DataFrame(results).T
print("\n", results_df[['Accuracy', 'Precision', 'Recall']])

LINEAR   -> Accuracy: 0.81, Precision: 0.76, Recall: 0.92
POLY     -> Accuracy: 0.93, Precision: 0.89, Recall: 0.97
RBF      -> Accuracy: 0.99, Precision: 0.97, Recall: 1.0
SIGMOID  -> Accuracy: 0.75, Precision: 0.72, Recall: 0.82

          Accuracy Precision    Recall
linear   0.814634   0.76378   0.92381
poly     0.926829  0.894737  0.971429
rbf      0.985366  0.972222       1.0
sigmoid  0.746341  0.722689  0.819048


# Problem 5: Customer Churn Prediction
### Dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

Objective
- Predict whether a customer will churn.

Tasks
- Handle missing values.
- Encode categorical columns.
- Scale numerical columns.
- Train SVM.
- Tune hyperparameters.


In [7]:
# load dataset
df_churn = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df_churn['TotalCharges'] = pd.to_numeric(df_churn['TotalCharges'], errors='coerce')

X = df_churn.drop(['customerID', 'Churn'], axis = 1)
y = df_churn['Churn'].map({'Yes': 1, 'No': 0})


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

# Handle missing values.
# Encode categorical columns.
# Scale numerical columns.
num_cols = X.select_dtypes(include = ['int64', 'float64']).columns
cat_cols = X.select_dtypes(include = 'object').columns

num_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'median')),
        ('scaler', StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer(strategy = 'most_frequent')),
        ('encoder', OneHotEncoder(sparse_output = False, drop = 'first', handle_unknown = 'ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ]
)

# Train SVM.
svm_mod = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', SVC())
    ]
)

grid_par = [
    {'model__kernel': ['linear'], 'model__C': [0.001, 0.01, 1]},
    {'model__kernel': ['rbf'], 'model__C': [0.1, 1, 10], 'model__gamma': ['scale', 0.01, 0.1]}
]

grid_search_svm = GridSearchCV(
    estimator = svm_mod,
    param_grid = grid_par,
    cv = 3
)

grid_search_svm.fit(X_train, y_train)
y_pred = grid_search_svm.predict(X_test)

# Print evaluation metrics.
acc_sv = accuracy_score(y_test, y_pred)
prec_sv = precision_score(y_test, y_pred)
rec_sv = recall_score(y_test, y_pred)

print(f'Accuracy: {round(acc_sv, 2)}, Precision: {round(prec_sv, 2)}, Recall: {round(rec_sv, 2)}')

Accuracy: 0.8, Precision: 0.67, Recall: 0.46


# Problem 6: Credit Card Fraud Detection
### Dataset: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

Objective
- Detect fraudulent transactions.

Tasks
- Handle imbalanced data. ( there is a parameter in SVC to address this )
- Train SVM.

Use:
- Precision
- Recall
- F1-score


In [8]:
# load dataset
df_card = pd.read_csv('creditcard.csv')

X = df_card.drop('Class', axis=1)
y = df_card['Class']

# Split first (this was the original bug - it was missing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Reduce training size only (test set stays full and untouched)
train_df = X_train.copy()
train_df['Class'] = y_train

fraud = train_df[train_df['Class'] == 1]
non_fraud = train_df[train_df['Class'] == 0].sample(n=50000, random_state=42)

train_small = pd.concat([fraud, non_fraud])
X_train_small = train_small.drop('Class', axis=1)
y_train_small = train_small['Class']

# Preprocess
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols)
])

# Train SVM — handles imbalance via class_weight
svm_mod = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', SVC(kernel='rbf', class_weight='balanced'))
])

svm_mod.fit(X_train_small, y_train_small)
y_pred = svm_mod.predict(X_test)

# Evaluate on FULL test set
prec_sv = precision_score(y_test, y_pred)
rec_sv = recall_score(y_test, y_pred)
f1_sv = f1_score(y_test, y_pred)

print(f'Precision: {round(prec_sv, 2)}, Recall: {round(rec_sv, 2)}, F1 Score: {round(f1_sv, 2)}')

Precision: 0.19, Recall: 0.81, F1 Score: 0.31
